# Deep Agents (optional)

Assistant version 10: Plan, research offline documents and write a sourced outdoor-gear trends report.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra deep --group authoring`, then `uv run --extra deep --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## create_agent plus TodoListMiddleware; bounded plan execution

Predict the output before running. The function definitions below match `examples/01_plan_by_hand.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Add an explicit bounded plan to create_agent before using a deep-agent harness.

Run: uv run --extra agents python modules/17_deep_agents/examples/01_plan_by_hand.py --offline
Expected: The todo list is pending; a plan is not evidence that tasks ran.
Cost: At most three short model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    TodoListMiddleware,
    ToolCallLimitMiddleware,
)

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    tasks = [
        {"content": "Read synthetic research sources", "status": "pending"},
        {"content": "Write and critique a cited report", "status": "pending"},
    ]
    model = get_chat_model(
        settings=settings,
        responses=[
            tool_request("write_todos", {"todos": tasks}),
            "Plan created; research not yet performed.",
        ],
    )
    # KEY IDEA: TodoListMiddleware stores a plan; separate tool calls must perform the work.
    agent = create_agent(
        model,
        [],
        middleware=[
            TodoListMiddleware(),
            ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=1, exit_behavior="end"),
        ],
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Plan a synthetic outdoor-gear report."}]},
        {"recursion_limit": 30},
    )
    print_text(str(result.get("todos", [])))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Allowlisted virtual filesystem tools; path confinement; no shell

Predict the output before running. The function definitions below match `examples/02_virtual_files.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Use an allowlisted in-memory filesystem with no shell access.

Run: uv run --extra agents python modules/17_deep_agents/examples/02_virtual_files.py --offline
Expected: One source read, one virtual report, and rejected path traversal.
Cost: At most four short model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.research import VirtualFiles


def main() -> None:
    settings = lesson_settings()
    files = VirtualFiles()
    responses = [
        tool_request("read_research", {"path": "/sources/market_brief.md"}, "read"),
        tool_request(
            "write_report",
            {
                "path": "/reports/trends.md",
                "content": (
                    "Synthetic sample: 48 of 80 preferred repairable gear "
                    "[/sources/market_brief.md]."
                ),
            },
            "write",
        ),
        "Draft stored in virtual memory.",
    ]
    agent = create_agent(
        get_chat_model(settings=settings, responses=responses),
        files.tools(),
        middleware=[
            ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=2, exit_behavior="end"),
        ],
    )
    agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Read the market brief, then write a cited synthetic summary to "
                        "/reports/trends.md."
                    ),
                }
            ]
        },
        {"recursion_limit": 30},
    )
    # KEY IDEA: virtual file names never become host paths, even for valid writes.
    print_text(files.files.get("/reports/trends.md", "No report was written."))
    try:
        files.read("/sources/../../.env")
    except ValueError as error:
        print_text(str(error))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Specialized research and critique subagents; budgets

Predict the output before running. The function definitions below match `examples/03_subagents.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Delegate research and critique with isolated histories and finite budgets.

Run: uv run --extra agents python modules/17_deep_agents/examples/03_subagents.py --offline
Expected: Research returns a source; critique receives only the draft evidence.
Cost: At most four short model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.research import VirtualFiles


def main() -> None:
    settings = lesson_settings()
    files = VirtualFiles()
    researcher = create_agent(
        get_chat_model(
            settings=settings,
            responses=[
                tool_request("read_research", {"path": "/sources/customer_trends.md"}),
                "Synthetic evidence: clearer setup guidance matters.",
            ],
        ),
        files.tools()[:1],
        middleware=[
            ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=1, exit_behavior="end"),
        ],
    )
    result = researcher.invoke(
        {"messages": [{"role": "user", "content": "Read /sources/customer_trends.md."}]},
        {"recursion_limit": 30},
    )
    evidence = [m.content for m in result["messages"] if m.type == "tool"]
    critic = create_agent(
        get_chat_model(
            settings=settings,
            responses=["Keep the synthetic label; do not generalize 40 interviews to the market."],
        ),
        [],
        middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
    )
    # KEY IDEA: the critic gets evidence and a narrow question, not the researcher's history.
    critique = critic.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Critique generalization risk in this evidence: " + str(evidence),
                }
            ]
        }
    )
    print_text(str(critique["messages"][-1].content))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## create_deep_agent; planning, virtual filesystem and subagents; optional pre-1.0 SDK path

Predict the output before running. The function definitions below match `examples/04_deepagents_sdk.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Exercise the verified optional SDK with state-backed files and a named subagent.

Run: uv run --extra deep python modules/17_deep_agents/examples/04_deepagents_sdk.py --offline
Expected: Planning, delegation and a virtual file write run through the actual SDK.
Cost: At most five supervisor calls plus bounded child calls; no shell or web service. Offline
fixtures incur no API fee.
Requires: deep
"""

from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    TodoListMiddleware,
    ToolCallLimitMiddleware,
)

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    researcher = get_chat_model(
        settings=settings, responses=["Synthetic report only; no real market inference."]
    )
    responses = [
        tool_request(
            "write_todos",
            {"todos": [{"content": "Prepare synthetic scope note", "status": "in_progress"}]},
            "plan",
        ),
        tool_request(
            "task",
            {"subagent_type": "researcher", "description": "Write a synthetic-data scope note."},
            "delegate",
        ),
        tool_request(
            "write_file",
            {
                "file_path": "/reports/scope.md",
                "content": "Synthetic report only; no real market inference.",
            },
            "file",
        ),
        "Scope note saved in state; no host file was written.",
    ]
    # KEY IDEA: StateBackend has virtual state storage, not host filesystem or shell access.
    agent = create_deep_agent(
        model=get_chat_model(settings=settings, responses=responses),
        backend=StateBackend(),
        subagents=[
            {
                "name": "general-purpose",
                "description": "Return a manual-review note; no further delegation",
                "model": get_chat_model(settings=settings, responses=["Request manual review."]),
                "tools": [],
                "middleware": [ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
            },
            {
                "name": "researcher",
                "description": "Prepare scope notes",
                "system_prompt": "Use only synthetic evidence.",
                "model": researcher,
                "tools": [],
                "middleware": [ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
            },
        ],
        middleware=[
            TodoListMiddleware(),
            ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=3, exit_behavior="end"),
        ],
    )
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Plan a scope note, delegate it to researcher, then write "
                        "/reports/scope.md."
                    ),
                }
            ]
        },
        {"recursion_limit": 60},
    )
    print_text(str({"files": result.get("files", {}), "todos": result.get("todos", [])}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Tavily opt-in web search; offline synthetic research corpus fallback; provenance

Predict the output before running. The function definitions below match `examples/05_search_sources.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Use a synthetic offline search corpus; make Tavily an explicit opt-in.

Run: uv run --extra agents python modules/17_deep_agents/examples/05_search_sources.py --offline
Expected: Retrieved source paths and synthetic labels; no default web call.
Cost: No model calls. --web uses one Tavily search and may consume search credits. Offline
fixtures incur no API fee.
Requires: agents
"""

import argparse

from common.config import get_settings
from common.errors import run_cli
from common.pretty import print_text
from common.research import research_sources


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--offline", action="store_true")
    parser.add_argument(
        "--web",
        action="store_true",
        help="Opt in to one Tavily search; requires search extra and key",
    )
    args = parser.parse_args()
    settings = get_settings()
    if args.offline:
        settings = settings.model_copy(update={"course_offline": True})
    # KEY IDEA: absence of --web selects local evidence even with a live model profile.
    docs = research_sources("repairable gear tent setup trends", settings, web=args.web)
    for doc in docs:
        print_text(
            str(
                {
                    "source": doc.metadata["source"],
                    "synthetic": doc.metadata["synthetic"],
                    "text": doc.page_content,
                }
            )
        )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Plan, research offline documents and write a sourced outdoor-gear trends report

Predict the output before running. The function definitions below match `use_case/research_report.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Plan, read synthetic sources, critique a draft and save a validated report.

Run: uv run --extra agents python modules/17_deep_agents/use_case/research_report.py --offline
Expected: Source-labelled synthetic report saved in the runtime directory.
Cost: At most six main-agent calls plus one critique call. Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    TodoListMiddleware,
    ToolCallLimitMiddleware,
)

from common.config import ROOT
from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.research import VirtualFiles, validate_report


def main() -> None:
    settings = lesson_settings()
    files = VirtualFiles()
    report = (
        "Synthetic outdoor-gear research\n\n48 of 80 invented shoppers "
        "preferred repairable gear [/sources/market_brief.md].\n24 of 40 "
        "invented interviewees wanted clearer tent setup guides "
        "[/sources/customer_trends.md].\nThese samples cannot establish "
        "real market trends."
    )
    responses = [
        tool_request(
            "write_todos",
            {
                "todos": [
                    {"content": "Read and cite two synthetic sources", "status": "in_progress"}
                ]
            },
            "plan",
        ),
        tool_request("read_research", {"path": "/sources/market_brief.md"}, "source1"),
        tool_request("read_research", {"path": "/sources/customer_trends.md"}, "source2"),
        tool_request("write_report", {"path": "/reports/trends.md", "content": report}, "write"),
        "Draft ready for critique.",
    ]
    agent = create_agent(
        get_chat_model(settings=settings, responses=responses),
        files.tools(),
        middleware=[
            TodoListMiddleware(),
            ModelCallLimitMiddleware(run_limit=6, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=4, exit_behavior="end"),
        ],
    )
    agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Plan, read market_brief.md and customer_trends.md in /sources/, "
                        "then write a cited synthetic report to /reports/trends.md."
                    ),
                }
            ]
        },
        {"recursion_limit": 60},
    )
    draft = files.read("/reports/trends.md")
    citations = validate_report(draft, files)
    # KEY IDEA: source validation is code; a critic's opinion does not prove factuality.
    critique = get_chat_model(
        settings=settings,
        responses=["Synthetic scope is explicit; check exact figures against the cited files."],
    ).invoke(
        "Critique this report against its evidence: "
        + draft
        + str({p: files.read(p) for p in citations})
    )
    output = ROOT / settings.runtime_dir / "research-report"
    output.mkdir(parents=True, exist_ok=True)
    (output / "report.md").write_text(draft, encoding="utf-8")
    (output / "critique.txt").write_text(str(critique.content), encoding="utf-8")
    print_text(str({"sources": citations, "output": str(output)}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/17_deep_agents/tests` for the supplied solution checks.